In [14]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

In [15]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [16]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [17]:
from dotenv import load_dotenv

In [18]:
model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
)

In [19]:
vector_store = Chroma(
    embedding_function=model,
    persist_directory="chroma_db",
    collection_name="sample"
)

### add documents

In [20]:
vector_store.add_documents(docs)

['5abbf7a0-4e5d-4c7b-a62c-60a8421916ea',
 'f6e5667c-962b-4e4b-8acb-6c1c66711303',
 '69bcc1c4-5713-49fc-8c9c-3a9ebab5cb46',
 'ad8f5fb8-9fb9-4481-9aba-6fffb1920698',
 '449387b3-580a-44b1-be8c-8e356bb5996b']

In [21]:
vector_store.get(include=['embeddings', 'documents', 'metadatas']) # should be metadatas not metadata

{'ids': ['7fe4f941-3751-4b52-bf22-3b88456eaf95',
  'e758ac76-69e9-4789-a163-93f30ef4dbc3',
  '0e234034-e414-4a2f-9c24-29d5a14ffeab',
  '03ea6536-5020-4fa2-b4e8-ed81a23c1039',
  '132cdcd8-2af4-456a-9125-7c990f384869',
  'ce607211-6b5d-4519-8c94-634a1b3960d2',
  'c954cf1b-85b2-41f6-8bc4-1dece5cdc682',
  '873fbed3-5191-410a-a858-63c0f64fc933',
  '6f1cde93-e6b3-4c84-ad06-e016692454d1',
  '287d580c-25c7-4287-a8ac-54198260fd5b',
  '5abbf7a0-4e5d-4c7b-a62c-60a8421916ea',
  'f6e5667c-962b-4e4b-8acb-6c1c66711303',
  '69bcc1c4-5713-49fc-8c9c-3a9ebab5cb46',
  'ad8f5fb8-9fb9-4481-9aba-6fffb1920698',
  '449387b3-580a-44b1-be8c-8e356bb5996b'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        ...,
        [-0.01358705, -0.0035

### search documents

In [22]:
# search document
vector_store.similarity_search(
    query="Who among these are a bowler",
    k=2
)

[Document(id='03ea6536-5020-4fa2-b4e8-ed81a23c1039', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='ad8f5fb8-9fb9-4481-9aba-6fffb1920698', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [23]:
# search document with score
vector_store.similarity_search_with_score(
    query="Who among these are a bowler",
    k=2
)

[(Document(id='03ea6536-5020-4fa2-b4e8-ed81a23c1039', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6391493082046509),
 (Document(id='ad8f5fb8-9fb9-4481-9aba-6fffb1920698', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6391493082046509)]

In [24]:

# We can also filer on metadata
vector_store.similarity_search_with_score(
    query="Who among these are a bowler",
    k=2,
    filter={"team":"Chennai Super Kings"}
)

[(Document(id='132cdcd8-2af4-456a-9125-7c990f384869', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.6609402894973755),
 (Document(id='449387b3-580a-44b1-be8c-8e356bb5996b', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.6609402894973755)]

### update document

In [27]:
updated_doc1 = Document(
        page_content="Virat Kohli the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistency",
        metadata={"team": "Royal Challengers Bangalore"}
    )

vector_store.update_document(document_id='5abbf7a0-4e5d-4c7b-a62c-60a8421916ea', document=updated_doc1)

In [ ]:
vector_store.get(include=["embeddings","documents","metadatas"])

{'ids': ['7fe4f941-3751-4b52-bf22-3b88456eaf95',
  'e758ac76-69e9-4789-a163-93f30ef4dbc3',
  '0e234034-e414-4a2f-9c24-29d5a14ffeab',
  '03ea6536-5020-4fa2-b4e8-ed81a23c1039',
  '132cdcd8-2af4-456a-9125-7c990f384869',
  'ce607211-6b5d-4519-8c94-634a1b3960d2',
  'c954cf1b-85b2-41f6-8bc4-1dece5cdc682',
  '873fbed3-5191-410a-a858-63c0f64fc933',
  '6f1cde93-e6b3-4c84-ad06-e016692454d1',
  '287d580c-25c7-4287-a8ac-54198260fd5b',
  '5abbf7a0-4e5d-4c7b-a62c-60a8421916ea',
  'f6e5667c-962b-4e4b-8acb-6c1c66711303',
  '69bcc1c4-5713-49fc-8c9c-3a9ebab5cb46',
  'ad8f5fb8-9fb9-4481-9aba-6fffb1920698',
  '449387b3-580a-44b1-be8c-8e356bb5996b'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        ...,
        [-0.01358705, -0.0035

In [34]:
vector_store.get()

{'ids': ['7fe4f941-3751-4b52-bf22-3b88456eaf95',
  'e758ac76-69e9-4789-a163-93f30ef4dbc3',
  '0e234034-e414-4a2f-9c24-29d5a14ffeab',
  '03ea6536-5020-4fa2-b4e8-ed81a23c1039',
  '132cdcd8-2af4-456a-9125-7c990f384869',
  'ce607211-6b5d-4519-8c94-634a1b3960d2',
  'c954cf1b-85b2-41f6-8bc4-1dece5cdc682',
  '873fbed3-5191-410a-a858-63c0f64fc933',
  '6f1cde93-e6b3-4c84-ad06-e016692454d1',
  '287d580c-25c7-4287-a8ac-54198260fd5b',
  '5abbf7a0-4e5d-4c7b-a62c-60a8421916ea',
  'f6e5667c-962b-4e4b-8acb-6c1c66711303',
  '69bcc1c4-5713-49fc-8c9c-3a9ebab5cb46',
  'ad8f5fb8-9fb9-4481-9aba-6fffb1920698',
  '449387b3-580a-44b1-be8c-8e356bb5996b'],
 'embeddings': None,
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm 

In [35]:
#delete document
vector_store.delete(ids=["7fe4f941-3751-4b52-bf22-3b88456eaf95"])

In [36]:
vector_store.get()

{'ids': ['e758ac76-69e9-4789-a163-93f30ef4dbc3',
  '0e234034-e414-4a2f-9c24-29d5a14ffeab',
  '03ea6536-5020-4fa2-b4e8-ed81a23c1039',
  '132cdcd8-2af4-456a-9125-7c990f384869',
  'ce607211-6b5d-4519-8c94-634a1b3960d2',
  'c954cf1b-85b2-41f6-8bc4-1dece5cdc682',
  '873fbed3-5191-410a-a858-63c0f64fc933',
  '6f1cde93-e6b3-4c84-ad06-e016692454d1',
  '287d580c-25c7-4287-a8ac-54198260fd5b',
  '5abbf7a0-4e5d-4c7b-a62c-60a8421916ea',
  'f6e5667c-962b-4e4b-8acb-6c1c66711303',
  '69bcc1c4-5713-49fc-8c9c-3a9ebab5cb46',
  'ad8f5fb8-9fb9-4481-9aba-6fffb1920698',
  '449387b3-580a-44b1-be8c-8e356bb5996b'],
 'embeddings': None,
 'documents': ["Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah i